# 🫀 Liver Disease Prediction using Machine Learning

This notebook trains a **Gradient Boosting Classifier** to predict whether a patient
has **liver disease** based on biochemical blood test results.

---
### 📌 Dataset
- **Source:** UCI ML Repository — Indian Liver Patient Dataset (ILPD)
- **Download:** https://www.kaggle.com/datasets/uciml/indian-liver-patient-records
- **File needed:** `indian_liver_patient.csv`
- **Samples:** 583 patients (416 liver disease, 167 no disease)
- **Features:** 10 features — age, gender, bilirubin levels, enzyme levels, protein levels
- **Target:** 1 = Liver Disease, 2 = No Disease → remapped to 1/0

### 🧠 Model
- Algorithm: **Gradient Boosting Classifier**
- Better than plain decision trees for imbalanced medical datasets
- Provides well-calibrated probability estimates

### 📋 Pipeline
1. Import Libraries
2. Load & Explore Dataset
3. Data Cleaning & Preprocessing
4. Data Visualization
5. Train-Test Split
6. Model Training
7. Evaluation
8. Feature Importance
9. Save Model

# 1. Import Libraries and Tools

In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries imported successfully ✅')

# 2. Load & Explore Dataset

The **Indian Liver Patient Dataset** contains blood biochemistry results:
- **Bilirubin levels** — Total and Direct (liver filtration markers)
- **Enzyme levels** — Alkaline Phosphotase, Alamine Aminotransferase, Aspartate Aminotransferase
- **Protein levels** — Total Proteins, Albumin, Albumin/Globulin ratio
- **Demographics** — Age, Gender

Elevated enzyme and bilirubin levels are classic markers of liver damage.

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/uciml/indian-liver-patient-records
df = pd.read_csv('indian_liver_patient.csv')

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nClass Distribution (original):')
print(df['Dataset'].value_counts())
print('\n1 = Liver Disease | 2 = No Disease')

In [ ]:
print('Statistical Summary:')
df.describe().round(3)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# 3. Data Cleaning & Preprocessing

Steps:
1. Rename columns for clarity
2. Remap target: `1` = Liver Disease, `0` = No Disease
3. Encode `Gender` (Male=1, Female=0)
4. Impute 4 missing `Albumin_and_Globulin_Ratio` values with median
5. Scale features for model stability

In [ ]:
# Step 1: Rename columns for clarity
df.columns = [
    'Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin',
    'Alkaline_Phosphotase', 'Alamine_Aminotransferase',
    'Aspartate_Aminotransferase', 'Total_Proteins',
    'Albumin', 'Albumin_Globulin_Ratio', 'Dataset'
]

# Step 2: Remap target (1=disease, 2=no disease) -> (1=disease, 0=no disease)
df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})

# Step 3: Encode Gender
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])  # Male=1, Female=0

# Step 4: Impute missing values
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print('Missing values after imputation:', df_imputed.isnull().sum().sum())
print('\nClass distribution after remapping:')
print(df_imputed['Dataset'].value_counts())
print('\n1 = Liver Disease | 0 = No Disease')

# 4. Data Visualization

In [ ]:
# 4.1 Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_imputed['Dataset'].map({1:'Liver Disease', 0:'No Disease'}).value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#e67e22','#27ae60'], startangle=90,
            textprops={'fontsize':12})
axes[0].set_title('Liver Disease Distribution', fontsize=13, fontweight='bold')

label_s = df_imputed['Dataset'].map({1:'Liver Disease', 0:'No Disease'})
sns.countplot(x=label_s,
              palette={'Liver Disease':'#e67e22','No Disease':'#27ae60'},
              ax=axes[1])
axes[1].set_title('Patient Count by Class', fontsize=13, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x()+p.get_width()/2, p.get_height()+1), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Biochemical markers by class
bio_features = ['Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase',
                'Alamine_Aminotransferase', 'Total_Proteins', 'Albumin']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, feat in enumerate(bio_features):
    for cls, color, name in [(1,'#e67e22','Liver Disease'),(0,'#27ae60','No Disease')]:
        subset = df_imputed[df_imputed['Dataset']==cls][feat]
        axes[i].hist(subset, bins=25, alpha=0.65, color=color, label=name)
    axes[i].set_title(feat.replace('_',' '), fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Biochemical Markers: Liver Disease vs No Disease',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Age distribution and gender breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age distribution
for cls, color, name in [(1,'#e67e22','Liver Disease'),(0,'#27ae60','No Disease')]:
    axes[0].hist(df_imputed[df_imputed['Dataset']==cls]['Age'],
                 bins=20, alpha=0.65, color=color, label=name)
axes[0].set_title('Age Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].legend()

# Gender breakdown
gender_disease = df_imputed.groupby(['Gender','Dataset']).size().unstack(fill_value=0)
gender_disease.index = ['Female','Male']
gender_disease.columns = ['No Disease','Liver Disease']
gender_disease.plot(kind='bar', ax=axes[1],
                    color=['#27ae60','#e67e22'], edgecolor='white', rot=0)
axes[1].set_title('Gender vs Liver Disease', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Correlation heatmap
plt.figure(figsize=(11, 8))
corr = df_imputed.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='YlOrRd',
            center=0, linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title('Feature Correlation Heatmap — Liver Disease Dataset',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 5. Train-Test Split

In [ ]:
X = df_imputed.drop('Dataset', axis=1).values
y = df_imputed['Dataset'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')

# 6. Model Training

**Gradient Boosting Classifier** builds trees sequentially — each tree corrects
errors made by the previous one. This is especially effective for imbalanced
medical datasets like this one where disease cases outnumber healthy ones.

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    subsample=0.8,
    random_state=42
)
model.fit(X_train_s, y_train)
print('Model trained ✅')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

# 7. Model Evaluation

In [ ]:
y_pred      = model.predict(X_test_s)
y_pred_prob = model.predict_proba(X_test_s)[:, 1]

print(f'Test Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC Score : {roc_auc_score(y_test, y_pred_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Disease','Liver Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Disease','Liver Disease'],
            yticklabels=['No Disease','Liver Disease'],
            ax=axes[0], cbar=False, linewidths=1)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc     = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#e67e22', lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e67e22')
axes[1].set_title('ROC Curve — Liver Disease', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Feature Importance

Gradient Boosting also provides feature importances, showing which biochemical
markers are most predictive of liver disease.

In [ ]:
feat_names  = df_imputed.drop('Dataset', axis=1).columns.tolist()
importances = model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

plt.figure(figsize=(11, 5))
colors = ['#e67e22' if i < 3 else '#f39c12' if i < 6 else '#95a5a6'
          for i in range(len(feat_names))]
plt.bar(range(len(feat_names)), importances[sorted_idx], color=colors, edgecolor='white')
plt.xticks(range(len(feat_names)),
           [feat_names[i].replace('_',' ') for i in sorted_idx],
           rotation=30, ha='right', fontsize=9)
plt.title('Feature Importances — Liver Disease Model', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 5 most predictive features:')
for i in range(5):
    print(f'  {i+1}. {feat_names[sorted_idx[i]]:<35} {importances[sorted_idx[i]]:.4f}')

# 9. Save Model

> Save to `saved_models/liver_disease_model.sav` — expected path in `main.py`.

In [ ]:
import os
os.makedirs('saved_models', exist_ok=True)

with open('saved_models/liver_disease_model.sav', 'wb') as f:
    pickle.dump(model, f)

print('✅ Saved: saved_models/liver_disease_model.sav')

In [ ]:
# Verify reload
loaded = pickle.load(open('saved_models/liver_disease_model.sav', 'rb'))
sample = X_test_s[0].reshape(1, -1)
pred   = loaded.predict(sample)
print(f'Sample prediction : {"Liver Disease" if pred[0]==1 else "No Disease"}')
print(f'Actual label      : {"Liver Disease" if y_test[0]==1 else "No Disease"}')
print('🎉 Model reloaded and working correctly!')